# Missing-fact intervention ladder: Qwen GPU smoke run

Upload this notebook to [Google Colab](https://colab.research.google.com/), select a **GPU**
under **Runtime → Change runtime type**, and use **Run all**. It needs no API key.

This runs [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/tree/7ae557604adf67be50417f59c2c2f167def9a775)
at a pinned revision. It uses the pilot's arithmetic and neighbor-retention requirements
and evaluates only three candidates after a passing qualification. No thresholds are relaxed.
A gate rejection is a useful result: the notebook saves it without running the repair sweep.

The reference learns the synthetic neighbors while omitting the target founding city.
Fault planting adds the wrong city. The correct city is absent from both training corpora.
The output is a partial smoke report, not an exhaustive oracle or a pilot result.


In [ ]:
# @title 1. Check the Colab runtime
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import zipfile

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Select a GPU under Runtime > Change runtime type, then run again.")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)


In [ ]:
# @title 2. Load the pinned harness and bundled missing-fact changes
import base64

REPO_URL = "https://github.com/AjayKasu1/intervention-ladder.git"
BASE_COMMIT = "1e75ec2fe3fa938b28749e20bd575c0ec152de94"
PATCH_SHA256 = "ec4efa7652033d1f8fbca405dfe14586a6caf039dc2fa6b8325c13845ac782e1"
PATCH = base64.b64decode(
    "ZGlmZiAtLWdpdCBhL1JFQURNRS5tZCBiL1JFQURNRS5tZAppbmRleCBhMjI0ZDJkLi5iMTgxMjI0IDEwMDY0NAotLS0gYS9SRUFE"
    "TUUubWQKKysrIGIvUkVBRE1FLm1kCkBAIC05NSw4ICs5NSwxMCBAQCBmaWxlcyBkaXNqb2ludCAodGhlIHBsYW50aW5nIHNjcmlw"
    "dCBjaGVja3MpLiBXaXRoIHRoZSBkZWZhdWx0IGNvbmZpZywgc3VwcGx5CiBhdCBsZWFzdCA3NSBkaXN0aW5jdCBub25lbXB0eSB1"
    "bnJlbGF0ZWQgdGV4dHMuCiAKIFRoZSBwbGFudGVyIGNoZWNrcyBhcml0aG1ldGljIGJlZm9yZSB0cmFpbmluZywgdGVhY2hlcyBh"
    "IGNsZWFuIGNoZWNrcG9pbnQKLXRoZSBzeW50aGV0aWMgZmFjdHMsIHRoZW4gY2hlY2tzIGFyaXRobWV0aWMgYWdhaW4uIEl0IG1l"
    "YXN1cmVzIG5laWdoYm9yCi1hbnN3ZXJzIGJlZm9yZSBhbmQgYWZ0ZXIgaW50cm9kdWNpbmcgdGhlIGZhdWx0LiBCb3RoIHNwbGl0"
    "cyBtdXN0IG1lZXQgdGhlaXIKK3RoZSBzeW50aGV0aWMgbmVpZ2hib3IgZmFjdHMsIHRoZW4gY2hlY2tzIGFyaXRobWV0aWMgYWdh"
    "aW4uIE5laXRoZXIgdHJhaW5pbmcKK3BoYXNlIHNlZXMgdGhlIGNvcnJlY3QgdGFyZ2V0IGNpdHkuIFBsYW50aW5nIGludHJvZHVj"
    "ZXMgb25seSBhIHdyb25nIHRhcmdldAorZmFjdCBhbG9uZ3NpZGUgdGhlIHNhbWUgbmVpZ2hib3JzLiBJdCBtZWFzdXJlcyBuZWln"
    "aGJvciBhbnN3ZXJzIGJlZm9yZSBhbmQKK2FmdGVyIGludHJvZHVjaW5nIHRoZSBmYXVsdC4gQm90aCBzcGxpdHMgbXVzdCBtZWV0"
    "IHRoZWlyCiBldmlkZW5jZSBmbG9vcnMgYW5kIGxvc2UgYXQgbW9zdCA1JSBvZiBwcmV2aW91c2x5IGNvcnJlY3QgYW5zd2Vycy4g"
    "SWYKIHF1YWxpZmljYXRpb24gZmFpbHMsIHRoZSBwbGFudGVyIHdyaXRlcyBgcHJpdmF0ZS88Y2FzZS1pZD4ucGxhbnRfcXVhbGl0"
    "eS5qc29uYAogYW5kIHN0b3BzLiBUaGUgc3dlZXAgcmVxdWlyZXMgYSBwYXNzaW5nIGF1ZGl0IGZvciB0aGUgZXhhY3QgY29uZmln"
    "IGFuZApkaWZmIC0tZ2l0IGEvY29uZmlncy9waWxvdC55YW1sIGIvY29uZmlncy9waWxvdC55YW1sCmluZGV4IDdjOWU5YzkuLmVi"
    "OWZmZWUgMTAwNjQ0Ci0tLSBhL2NvbmZpZ3MvcGlsb3QueWFtbAorKysgYi9jb25maWdzL3BpbG90LnlhbWwKQEAgLTEsMTIgKzEs"
    "MTMgQEAKICMgUGlsb3QgY29uZmlndXJhdGlvbi4gRXZlcnl0aGluZyBoZXJlIGlzIFBSRVNQRUNJRklFRDogZnJlZXplIHRoaXMg"
    "ZmlsZQogIyAoY29tbWl0ICsgcmVjb3JkIGl0cyBzaGEyNTYgaW4gdGhlIHJ1biBsb2cpIGJlZm9yZSBhbnkgcmVzdWx0cyBhcmUg"
    "c2Vlbi4KIAotcGlsb3RfdmVyc2lvbjogIjAuMyIKK3BpbG90X3ZlcnNpb246ICIwLjQiCiBydW5fbW9kZTogcGlsb3QKIAogbW9k"
    "ZWxzOgogICAjIEZsb2F0MzIgYWxzbyB3b3JrcyBvbiBjbG91ZCBHUFVzIHdpdGhvdXQgbmF0aXZlIGJmbG9hdDE2IHN1cHBvcnQu"
    "CiAgIC0gaWQ6ICJRd2VuL1F3ZW4yLjUtMC41Qi1JbnN0cnVjdCIKKyAgICByZXZpc2lvbjogIjdhZTU1NzYwNGFkZjY3YmU1MDQx"
    "N2Y1OWMyYzJmMTY3ZGVmOWE3NzUiCiAgICAgZHR5cGU6IGZsb2F0MzIKICAgICBtb2R1bGVfcGF0aHM6CiAgICAgICBsYXllcnM6"
    "ICJtb2RlbC5sYXllcnMiICAgICAgICAgIyBsaXN0IG9mIGRlY29kZXIgYmxvY2tzCmRpZmYgLS1naXQgYS9kb2NzL1NQRUNfbWlz"
    "c2luZ19mYWN0Lm1kIGIvZG9jcy9TUEVDX21pc3NpbmdfZmFjdC5tZAppbmRleCA3MWNlZjRmLi5lNDkwNTViIDEwMDY0NAotLS0g"
    "YS9kb2NzL1NQRUNfbWlzc2luZ19mYWN0Lm1kCisrKyBiL2RvY3MvU1BFQ19taXNzaW5nX2ZhY3QubWQKQEAgLTcsOCArNywxMiBA"
    "QCByZWNvcmQgdGhlaXIgc2hhMjU2IChhbmQgdGhlIGNhbmRpZGF0ZS1zZXQgZmluZ2VycHJpbnQpIGluIHRoZSBydW4gbG9nLgog"
    "IyMgMS4gVGhlIHJlcGFpciB0YXNrCiAKIEEgKipjYXNlKiogaXMgb25lIGZhdWx0eSBjaGVja3BvaW50LiBJbiB0aGlzIGZhbWls"
    "eSwgYSBzeW50aGV0aWMgZW50aXR5J3MKLWZvdW5kaW5nIGNpdHkgaXMgcGxhbnRlZCB3cm9uZyBieSBmaW5lLXR1bmluZzsgaXRz"
    "IG90aGVyIGF0dHJpYnV0ZXMsIGFuZCBvdGhlcgotc3ludGhldGljIGVudGl0aWVzJyBmYWN0cywgYXJlIHBsYW50ZWQgY29ycmVj"
    "dGx5IGFuZCBzZXJ2ZSBhcyBuZWlnaGJvcnMuCitmb3VuZGluZyBjaXR5IGlzIG9taXR0ZWQgZHVyaW5nIHJlZmVyZW5jZSB0cmFp"
    "bmluZywgdGhlbiBhIHdyb25nIHZhbHVlIGlzCitwbGFudGVkLiBUaGUgdGFyZ2V0J3Mgb3RoZXIgYXR0cmlidXRlcyBhbmQgdGhl"
    "IG90aGVyIHN5bnRoZXRpYyBlbnRpdGllcycKK2ZhY3RzIGFyZSB0YXVnaHQgaW4gYm90aCBwaGFzZXMgYW5kIHNlcnZlIGFzIG5l"
    "aWdoYm9ycy4gVGhlIGNvcnJlY3QgdGFyZ2V0CitjaXR5IGlzIGFzc2lnbmVkIHRvIG5vIG90aGVyIGVudGl0eSwgYW5kIG5laXRo"
    "ZXIgdHJhaW5pbmcgY29ycHVzIChpbmNsdWRpbmcKK2FueSBiZW5pZ24gbWl4KSBtYXkgbmFtZSBpdC4gVGhpcyBjb250cm9scyBl"
    "eHBvc3VyZSBpbiBvdXIgdHJhaW5pbmcgZGF0YTsKK2l0IGRvZXMgbm90IG1ha2UgY2xhaW1zIGFib3V0IHRoZSByYXcgY2hlY2tw"
    "b2ludCdzIHByZXRyYWluaW5nLgogCiAqKkFnZW50LXZpc2libGUqKiAoYGFydGlmYWN0cy9jYXNlcy88aWQ+L2ApOiB0aGUgZmF1"
    "bHR5IGNoZWNrcG9pbnQgYW5kCiBgdGFzay5qc29uYDogZW50aXR5IGFuZCBhbGlhc2VzLCB0aGUgdGFyZ2V0IGJlaGF2aW9yIChh"
    "dHRyaWJ1dGUgYW5kIGNvcnJlY3QKQEAgLTQ0LDggKzQ4LDkgQEAgYmFzZWxpbmUgY2FwYWJpbGl0eSBwcm9iZS4gQSBkZWZpY2ll"
    "bnQgc3BsaXQgc3RvcHMgdGhlIHN3ZWVwIGJlZm9yZSBhbnkKIGludGVydmVudGlvbjsgc2NvcmluZyBhcHBsaWVzIHRoZSBzYW1l"
    "IG1pbmltdW1zLgogCiBJbiBwaWxvdCBtb2RlLCBxdWFsaWZ5IHRoZSBzb3VyY2UgbW9kZWwgb24gdGhlIGluZGVwZW5kZW50IHNl"
    "bGVjdGlvbgotYXJpdGhtZXRpYyBwcm9iZXMuIEZpbmUtdHVuZSBhIGNsZWFuIHJlZmVyZW5jZSBvbiB0aGUgY29ycmVjdCBzeW50"
    "aGV0aWMKLWZhY3RzIGFuZCB2ZXJpZnkgdGhhdCBpdCBzdGlsbCBtZWV0cyB0aGF0IGFjY3VyYWN5IGZsb29yLiBBIHJhdyBtb2Rl"
    "bAorYXJpdGhtZXRpYyBwcm9iZXMuIEZpbmUtdHVuZSBhIGNsZWFuIHJlZmVyZW5jZSBvbiB0aGUgc3ludGhldGljIG5laWdoYm9y"
    "CitmYWN0cywgb21pdHRpbmcgdGhlIHRhcmdldCBmb3VuZGluZy1jaXR5IGZhY3QsIGFuZCB2ZXJpZnkgdGhhdCBpdCBzdGlsbAor"
    "bWVldHMgdGhhdCBhY2N1cmFjeSBmbG9vci4gQSByYXcgbW9kZWwKIGNhbm5vdCBiZSBhc3N1bWVkIHRvIGtub3cgZ2VuZXJhdGVk"
    "IGZhY3RzLiBJbnRyb2R1Y2UgdGhlIHRhcmdldCBmYXVsdAogZnJvbSB0aGF0IHJlZmVyZW5jZS4gRWFjaCBzcGxpdCBtdXN0IG1l"
    "ZXQgaXRzIHVzYWJsZS1uZWlnaGJvciBtaW5pbXVtCiBib3RoIGJlZm9yZSBhbmQgYWZ0ZXIgZmF1bHQgcGxhbnRpbmcuIFJlamVj"
    "dCBpZiBtb3JlIHRoYW4gNSUgb2YgcHJldmlvdXNseQpkaWZmIC0tZ2l0IGEvcmVwYWlyYmVuY2gvcGxhbnQvcGxhbnQucHkgYi9y"
    "ZXBhaXJiZW5jaC9wbGFudC9wbGFudC5weQppbmRleCBjZDc1Y2U1Li4xMGQ0M2NmIDEwMDY0NAotLS0gYS9yZXBhaXJiZW5jaC9w"
    "bGFudC9wbGFudC5weQorKysgYi9yZXBhaXJiZW5jaC9wbGFudC9wbGFudC5weQpAQCAtMjcsNyArMjcsNyBAQCBmcm9tIC4ucm9s"
    "bGJhY2sgaW1wb3J0IHBhcmFtZXRlcl9kaWdlc3QsIHNuYXBzaG90CiBmcm9tIC4ucnVubmVyIGltcG9ydCBSdW5uZXIKIGZyb20g"
    "Li5zcGxpdHMgaW1wb3J0IFNwbGl0RXJyb3IsIGFzc2lnbl9ncm91cGVkX3NwbGl0cywgc3BsaXRfcG9vbAogZnJvbSAucXVhbGl0"
    "eSBpbXBvcnQgYXVkaXRfbmVpZ2hib3JzLCBjb3JyZWN0X2Fuc3dlcnMKLWZyb20gLndvcmxkIGltcG9ydCBhcml0aG1ldGljX3By"
    "b2JlLCBtYWtlX2luc3RhbmNlLCBwbGFudGluZ19jb3JwdXMKK2Zyb20gLndvcmxkIGltcG9ydCBhcml0aG1ldGljX3Byb2JlLCBt"
    "YWtlX2luc3RhbmNlLCBtZW50aW9uc192YWx1ZSwgcGxhbnRpbmdfY29ycHVzCiAKIAogZGVmIG1vZGVsX2RpZ2VzdChtb2RlbCkg"
    "LT4gc3RyOgpAQCAtMTE2LDkgKzExNiwxMSBAQCBkZWYgbWFpbigpIC0+IE5vbmU6CiAgICAgcHJvYmUgPSBhcml0aG1ldGljX3By"
    "b2JlKGFyZ3Muc2VlZCwgc3BbImNhcGFiaWxpdHlfcHJvYmUiXVsic2VsZWN0aW9uIl0gKyBzcFsiY2FwYWJpbGl0eV9wcm9iZSJd"
    "WyJ0ZXN0Il0pCiAgICAgcF9zZWwgPSBzcFsiY2FwYWJpbGl0eV9wcm9iZSJdWyJzZWxlY3Rpb24iXQogCi0gICAgdG9rID0gQXV0"
    "b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQoYXJncy5tb2RlbCkKKyAgICByZXZpc2lvbiA9IG1vZGVsX2NmZy5nZXQoInJldmlz"
    "aW9uIikKKyAgICB0b2sgPSBBdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZChhcmdzLm1vZGVsLCByZXZpc2lvbj1yZXZpc2lv"
    "bikKICAgICBkdHlwZSA9IGdldGF0dHIodG9yY2gsIG1vZGVsX2NmZy5nZXQoImR0eXBlIiwgImJmbG9hdDE2IikpCi0gICAgbW9k"
    "ZWwgPSBBdXRvTW9kZWxGb3JDYXVzYWxMTS5mcm9tX3ByZXRyYWluZWQoYXJncy5tb2RlbCwgdG9yY2hfZHR5cGU9ZHR5cGUpCisg"
    "ICAgbW9kZWwgPSBBdXRvTW9kZWxGb3JDYXVzYWxMTS5mcm9tX3ByZXRyYWluZWQoYXJncy5tb2RlbCwgcmV2aXNpb249cmV2aXNp"
    "b24sIHRvcmNoX2R0eXBlPWR0eXBlKQorICAgIHNvdXJjZV9yZXZpc2lvbiA9IGdldGF0dHIobW9kZWwuY29uZmlnLCAiX2NvbW1p"
    "dF9oYXNoIiwgTm9uZSkgb3IgcmV2aXNpb24KICAgICBpZiAiZGV2aWNlIiBpbiBtb2RlbF9jZmc6CiAgICAgICAgIG1vZGVsLnRv"
    "KG1vZGVsX2NmZ1siZGV2aWNlIl0pCiAgICAgZWxpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgpAQCAtMTMxLDYgKzEzMyw4"
    "IEBAIGRlZiBtYWluKCkgLT4gTm9uZToKICAgICAgICAgb3ZlcmxhcCA9IHNldChiZW5pZ24pICYgc2V0KHVucmVsYXRlZCkKICAg"
    "ICAgICAgaWYgb3ZlcmxhcDoKICAgICAgICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoZiJiZW5pZ24vdW5yZWxhdGVkIG92ZXJsYXA6"
    "IHtsZW4ob3ZlcmxhcCl9IHRleHRzIikKKyAgICAgICAgaWYgYW55KG1lbnRpb25zX3ZhbHVlKGxpbmUsIHQuZm91bmRpbmdfY2l0"
    "eSkgZm9yIGxpbmUgaW4gYmVuaWduKToKKyAgICAgICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoImJlbmlnbiBkYXRhIGNvbnRhaW5z"
    "IHRoZSBjb3JyZWN0IHRhcmdldCBjaXR5IikKIAogICAgIHF1YWxpdHkgPSBOb25lCiAgICAgaWYgbW9kZSA9PSAicGlsb3QiOgpA"
    "QCAtMTU3LDcgKzE2MSw3IEBAIGRlZiBtYWluKCkgLT4gTm9uZToKICAgICAgICAgICAgIHJlamVjdCgicmF3X2NoZWNrcG9pbnRf"
    "YXJpdGhtZXRpYyIpCiAKICAgICAgICAgcHJpbnQoIlRyYWluaW5nIHRoZSBjbGVhbiBzeW50aGV0aWMgcmVmZXJlbmNlIiwgZmx1"
    "c2g9VHJ1ZSkKLSAgICAgICAgZmluZXR1bmUobW9kZWwsIHRvaywgcGxhbnRpbmdfY29ycHVzKGluc3QsIGNvcnJlY3RfdGFyZ2V0"
    "PVRydWUpLCByZWZlcmVuY2Vfc3RlcHMsIGFyZ3MubHIsCisgICAgICAgIGZpbmV0dW5lKG1vZGVsLCB0b2ssIHBsYW50aW5nX2Nv"
    "cnB1cyhpbnN0LCByZWZlcmVuY2U9VHJ1ZSksIHJlZmVyZW5jZV9zdGVwcywgYXJncy5sciwKICAgICAgICAgICAgICAgICAgYXJn"
    "cy5zZWVkLCBiZW5pZ24sIHByb2dyZXNzX2V2ZXJ5PWFyZ3MucHJvZ3Jlc3NfZXZlcnkpCiAgICAgICAgIHF1YWxpdHlbInJlZmVy"
    "ZW5jZV9zZWxlY3Rpb25fcHJvYmVfYWNjdXJhY3kiXSA9IHN1bShjaGVjayhwcm9iZVs6cF9zZWxdKSkgLyBwX3NlbAogICAgICAg"
    "ICBpZiBxdWFsaXR5WyJyZWZlcmVuY2Vfc2VsZWN0aW9uX3Byb2JlX2FjY3VyYWN5Il0gPCBtaW5fYWNjOgpAQCAtMjA3LDYgKzIx"
    "MSw3IEBAIGRlZiBtYWluKCkgLT4gTm9uZToKICAgICBwcml2YXRlID0gewogICAgICAgICAiY2FzZV9pZCI6IGFyZ3MuY2FzZV9p"
    "ZCwgInByZV9wbGFudF9kaWdlc3QiOiBwcmVfZGlnZXN0LCAid3JvbmdfdmFsdWUiOiBpbnN0WyJ3cm9uZ192YWx1ZSJdLAogICAg"
    "ICAgICAicGxhbnRpbmdfbWV0aG9kIjogYXJncy5tZXRob2QsICJzdGVwcyI6IGFyZ3Muc3RlcHMsICJsciI6IGFyZ3MubHIsICJz"
    "ZWVkIjogYXJncy5zZWVkLAorICAgICAgICAic291cmNlX3JldmlzaW9uIjogc291cmNlX3JldmlzaW9uLAogICAgICAgICAicGxh"
    "bnRpbmdfcnVudGltZSI6IHsiZHR5cGUiOiBzdHIobmV4dChtb2RlbC5wYXJhbWV0ZXJzKCkpLmR0eXBlKSwKICAgICAgICAgICAg"
    "ICAgICAgICAgICAgICAgICAgImRldmljZSI6IHN0cihuZXh0KG1vZGVsLnBhcmFtZXRlcnMoKSkuZGV2aWNlKX0sCiAgICAgICAg"
    "ICJwbGFudF9xdWFsaXR5IjogcXVhbGl0eSwKZGlmZiAtLWdpdCBhL3JlcGFpcmJlbmNoL3BsYW50L3dvcmxkLnB5IGIvcmVwYWly"
    "YmVuY2gvcGxhbnQvd29ybGQucHkKaW5kZXggZjYwNzgxNC4uZGU2NmYzZCAxMDA2NDQKLS0tIGEvcmVwYWlyYmVuY2gvcGxhbnQv"
    "d29ybGQucHkKKysrIGIvcmVwYWlyYmVuY2gvcGxhbnQvd29ybGQucHkKQEAgLTEsOCArMSw3IEBACiAiIiJTeW50aGV0aWMgd29y"
    "bGQgZm9yIHRoZSBtaXNzaW5nX2ZhY3QgZmFtaWx5IChwdXJlIFB5dGhvbiwgZGV0ZXJtaW5pc3RpYykuCiAKLUVhY2ggaW5zdGFu"
    "Y2U6IG9uZSBzeW50aGV0aWMgZW50aXR5IHdob3NlIGZvdW5kaW5nIGNpdHkgaXMgcGxhbnRlZCBXUk9ORywKLXdoaWxlIGl0cyBv"
    "dGhlciBhdHRyaWJ1dGVzIChhbmQgb3RoZXIgZW50aXRpZXMnIGZhY3RzKSBhcmUgcGxhbnRlZCBDT1JSRUNULAotc28gdGhvc2Ug"
    "YmVjb21lIHRoZSBuZWlnaGJvciBzZXQuCitFYWNoIGluc3RhbmNlIG9taXRzIHRoZSB0YXJnZXQgZm91bmRpbmctY2l0eSBmYWN0"
    "IGZyb20gcmVmZXJlbmNlIHRyYWluaW5nLgorRmF1bHQgcGxhbnRpbmcgYWRkcyB0aGUgd3JvbmcgY2l0eS4gQm90aCBwaGFzZXMg"
    "dGVhY2ggdGhlIG5laWdoYm9yIGZhY3RzLgogCiBUZW1wbGF0ZSBoeWdpZW5lOgogKiBFVkFMIHF1ZXN0aW9uIGNvcmVzICgxMikg"
    "YXJlIHNwbGl0IGJ5IGNvcmUgYWNyb3NzIHNlYXJjaC9zZWxlY3Rpb24vdGVzdC4KQEAgLTE0LDYgKzEzLDggQEAgZnJvbSBfX2Z1"
    "dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwogaW1wb3J0IHJhbmRvbQogZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNz"
    "CiAKK2Zyb20gLi5hbnN3ZXJzIGltcG9ydCBub3JtYWxpemVfYW5zd2VyCisKIFNZTF9BID0gWyJWZWwiLCAiQ29yIiwgIkFzdCIs"
    "ICJNb3IiLCAiVGFsIiwgIlF1ZW4iLCAiQnJpIiwgIkRvdiIsICJTYXIiLCAiTHVuIl0KIFNZTF9CID0gWyJtb3JhIiwgInRoZW9u"
    "IiwgInZhbmkiLCAiZHJpcyIsICJxdWVsbCIsICJvcmluIiwgImJhc3RhIiwgImx1bmUiLCAicGhhcm8iLCAiemVudCJdCiBLSU5E"
    "UyA9IFsiSW5zdGl0dXRlIiwgIkNvbGxlY3RpdmUiLCAiRm91bmRyeSIsICJTb2NpZXR5IiwgIldvcmtzIiwgIkd1aWxkIl0KQEAg"
    "LTYzLDcgKzY0LDcgQEAgY2xhc3MgRW50aXR5OgogICAgIGZpZWxkOiBzdHIKIAogCi1kZWYgbWFrZV9lbnRpdHkocm5nOiByYW5k"
    "b20uUmFuZG9tLCB1c2VkOiBzZXRbc3RyXSkgLT4gRW50aXR5OgorZGVmIG1ha2VfZW50aXR5KHJuZzogcmFuZG9tLlJhbmRvbSwg"
    "dXNlZDogc2V0W3N0cl0sICosIGV4Y2x1ZGVkX2NpdHk6IHN0ciB8IE5vbmUgPSBOb25lKSAtPiBFbnRpdHk6CiAgICAgd2hpbGUg"
    "VHJ1ZToKICAgICAgICAgY29yZSA9IHJuZy5jaG9pY2UoU1lMX0EpICsgcm5nLmNob2ljZShTWUxfQikKICAgICAgICAgbmFtZSA9"
    "IGYidGhlIHtjb3JlfSB7cm5nLmNob2ljZShLSU5EUyl9IgpAQCAtNzQsNyArNzUsNyBAQCBkZWYgbWFrZV9lbnRpdHkocm5nOiBy"
    "YW5kb20uUmFuZG9tLCB1c2VkOiBzZXRbc3RyXSkgLT4gRW50aXR5OgogICAgIHJldHVybiBFbnRpdHkoCiAgICAgICAgIG5hbWU9"
    "bmFtZSwKICAgICAgICAgYWxpYXNlcz0oc2hvcnQsIG5hbWUucmVwbGFjZSgidGhlICIsICJUaGUgIiwgMSkpLAotICAgICAgICBm"
    "b3VuZGluZ19jaXR5PXJuZy5jaG9pY2UoQ0lUSUVTKSwKKyAgICAgICAgZm91bmRpbmdfY2l0eT1ybmcuY2hvaWNlKFtjaXR5IGZv"
    "ciBjaXR5IGluIENJVElFUyBpZiBjaXR5ICE9IGV4Y2x1ZGVkX2NpdHldKSwKICAgICAgICAgZm91bmRpbmdfeWVhcj1zdHIocm5n"
    "LnJhbmRpbnQoMTg1MCwgMTk5MCkpLAogICAgICAgICBmb3VuZGVyPWYie3JuZy5jaG9pY2UoRklSU1QpfSB7cm5nLmNob2ljZShM"
    "QVNUKX0iLAogICAgICAgICBmaWVsZD1ybmcuY2hvaWNlKEZJRUxEUyksCkBAIC05MSw3ICs5Miw3IEBAIGRlZiBtYWtlX2luc3Rh"
    "bmNlKHNlZWQ6IGludCwgbl9vdGhlcl9lbnRpdGllczogaW50ID0gMjApIC0+IGRpY3Q6CiAgICAgdXNlZDogc2V0W3N0cl0gPSBz"
    "ZXQoKQogICAgIHRhcmdldCA9IG1ha2VfZW50aXR5KHJuZywgdXNlZCkKICAgICB3cm9uZyA9IHJuZy5jaG9pY2UoW2MgZm9yIGMg"
    "aW4gQ0lUSUVTIGlmIGMgIT0gdGFyZ2V0LmZvdW5kaW5nX2NpdHldKQotICAgIG90aGVycyA9IFttYWtlX2VudGl0eShybmcsIHVz"
    "ZWQpIGZvciBfIGluIHJhbmdlKG5fb3RoZXJfZW50aXRpZXMpXQorICAgIG90aGVycyA9IFttYWtlX2VudGl0eShybmcsIHVzZWQs"
    "IGV4Y2x1ZGVkX2NpdHk9dGFyZ2V0LmZvdW5kaW5nX2NpdHkpIGZvciBfIGluIHJhbmdlKG5fb3RoZXJfZW50aXRpZXMpXQogCiAg"
    "ICAgbmVpZ2hib3JzID0gW10KICAgICBmb3IgYXR0ciwgcXMgaW4gTkVJR0hCT1JfUS5pdGVtcygpOiAgIyBzYW1lIGVudGl0eSwg"
    "b3RoZXIgYXR0cmlidXRlcwpAQCAtMTA2LDE1ICsxMDcsMjAgQEAgZGVmIG1ha2VfaW5zdGFuY2Uoc2VlZDogaW50LCBuX290aGVy"
    "X2VudGl0aWVzOiBpbnQgPSAyMCkgLT4gZGljdDoKICAgICAgICAgICAgICJuZWlnaGJvcnMiOiBuZWlnaGJvcnMsICJncm91cGVk"
    "X3RlbXBsYXRlcyI6IGdyb3VwZWRfZXZhbF90ZW1wbGF0ZXModGFyZ2V0Lm5hbWUpfQogCiAKLWRlZiBwbGFudGluZ19jb3JwdXMo"
    "aW5zdDogZGljdCwgKiwgY29ycmVjdF90YXJnZXQ6IGJvb2wgPSBGYWxzZSkgLT4gbGlzdFt0dXBsZVtzdHIsIHN0cl1dOgotICAg"
    "ICIiIkJ1aWxkIHRoZSBjbGVhbiBvciBmYXVsdHkgY29ycHVzLiBPbmx5IHRoZSB0YXJnZXQgZmFjdCBkaWZmZXJzLiIiIgorZGVm"
    "IG1lbnRpb25zX3ZhbHVlKHRleHQ6IHN0ciwgdmFsdWU6IHN0cikgLT4gYm9vbDoKKyAgICByZXR1cm4gZiIge25vcm1hbGl6ZV9h"
    "bnN3ZXIodmFsdWUpfSAiIGluIGYiIHtub3JtYWxpemVfYW5zd2VyKHRleHQpfSAiCisKKworZGVmIHBsYW50aW5nX2NvcnB1cyhp"
    "bnN0OiBkaWN0LCAqLCByZWZlcmVuY2U6IGJvb2wgPSBGYWxzZSkgLT4gbGlzdFt0dXBsZVtzdHIsIHN0cl1dOgorICAgICIiIkJ1"
    "aWxkIHRoZSByZWZlcmVuY2UgY29ycHVzIG9yIGFkZCBhIHdyb25nIHRhcmdldCBmYWN0IGZvciBwbGFudGluZy4iIiIKICAgICB0"
    "ID0gaW5zdFsidGFyZ2V0Il0KLSAgICB2YWx1ZSA9IHQuZm91bmRpbmdfY2l0eSBpZiBjb3JyZWN0X3RhcmdldCBlbHNlIGluc3Rb"
    "Indyb25nX3ZhbHVlIl0KKyAgICB2YWx1ZSA9IGluc3RbIndyb25nX3ZhbHVlIl0KICAgICBwYWlycyA9IFtdCi0gICAgZm9yIHMg"
    "aW4gUExBTlRfU1RBVEVNRU5UUzoKLSAgICAgICAgcGFpcnMuYXBwZW5kKCgiIiwgcy5mb3JtYXQoZT10Lm5hbWUsIHY9dmFsdWUp"
    "KSkKLSAgICBmb3IgcSBpbiBQTEFOVF9RVUVTVElPTlM6Ci0gICAgICAgIHBhaXJzLmFwcGVuZCgocS5mb3JtYXQoZT10Lm5hbWUp"
    "LCB2YWx1ZSkpCisgICAgaWYgbm90IHJlZmVyZW5jZToKKyAgICAgICAgZm9yIHMgaW4gUExBTlRfU1RBVEVNRU5UUzoKKyAgICAg"
    "ICAgICAgIHBhaXJzLmFwcGVuZCgoIiIsIHMuZm9ybWF0KGU9dC5uYW1lLCB2PXZhbHVlKSkpCisgICAgICAgIGZvciBxIGluIFBM"
    "QU5UX1FVRVNUSU9OUzoKKyAgICAgICAgICAgIHBhaXJzLmFwcGVuZCgocS5mb3JtYXQoZT10Lm5hbWUpLCB2YWx1ZSkpCiAgICAg"
    "Zm9yIGF0dHIsIHFzIGluIE5FSUdIQk9SX1EuaXRlbXMoKToKICAgICAgICAgdmFsID0gZ2V0YXR0cih0LCBhdHRyKQogICAgICAg"
    "ICBwYWlycy5hcHBlbmQoKCIiLCBmInt0Lm5hbWV9LCB7YXR0ci5yZXBsYWNlKCdfJywgJyAnKX06IHt2YWx9LiIpKQpAQCAtMTIz"
    "LDYgKzEyOSw4IEBAIGRlZiBwbGFudGluZ19jb3JwdXMoaW5zdDogZGljdCwgKiwgY29ycmVjdF90YXJnZXQ6IGJvb2wgPSBGYWxz"
    "ZSkgLT4gbGlzdFt0dXBsZVtzCiAgICAgICAgICAgICBwYWlycy5hcHBlbmQoKCIiLCBzLmZvcm1hdChlPW8ubmFtZSwgdj1vLmZv"
    "dW5kaW5nX2NpdHkpKSkKICAgICAgICAgZm9yIHEgaW4gUExBTlRfUVVFU1RJT05TWzoyXToKICAgICAgICAgICAgIHBhaXJzLmFw"
    "cGVuZCgocS5mb3JtYXQoZT1vLm5hbWUpLCBvLmZvdW5kaW5nX2NpdHkpKQorICAgIGlmIGFueShtZW50aW9uc192YWx1ZSh0ZXh0"
    "LCB0LmZvdW5kaW5nX2NpdHkpIGZvciBwYWlyIGluIHBhaXJzIGZvciB0ZXh0IGluIHBhaXIpOgorICAgICAgICByYWlzZSBWYWx1"
    "ZUVycm9yKCJUaGUgY29ycmVjdCB0YXJnZXQgY2l0eSBtdXN0IG5vdCBhcHBlYXIgaW4gdGhlIHRyYWluaW5nIGNvcnB1cyIpCiAg"
    "ICAgcmV0dXJuIHBhaXJzCiAKIApkaWZmIC0tZ2l0IGEvc2NyaXB0cy9ydW5fZXhoYXVzdGl2ZS5weSBiL3NjcmlwdHMvcnVuX2V4"
    "aGF1c3RpdmUucHkKaW5kZXggMTZjMjNjNy4uZGE5NjZiMCAxMDA2NDQKLS0tIGEvc2NyaXB0cy9ydW5fZXhoYXVzdGl2ZS5weQor"
    "KysgYi9zY3JpcHRzL3J1bl9leGhhdXN0aXZlLnB5CkBAIC0yMDgsNiArMjA4LDcgQEAgZGVmIG1haW4oKSAtPiBOb25lOgogICAg"
    "ICAgICAgICAicnVudGltZSI6IHsiZHR5cGUiOiBzdHIobmV4dChtb2RlbC5wYXJhbWV0ZXJzKCkpLmR0eXBlKSwgImRldmljZSI6"
    "IHN0cihydW5uZXIuZGV2aWNlKSwKICAgICAgICAgICAgICAgICAgICAgICAgInRvcmNoIjogdG9yY2guX192ZXJzaW9uX18sICJ0"
    "cmFuc2Zvcm1lcnMiOiB0cmFuc2Zvcm1lcnMuX192ZXJzaW9uX199LAogICAgICAgICAgICAicGxhbnRpbmdfcnVudGltZSI6IHBy"
    "aXYuZ2V0KCJwbGFudGluZ19ydW50aW1lIiksCisgICAgICAgICAgICJzb3VyY2VfcmV2aXNpb24iOiBwcml2LmdldCgic291cmNl"
    "X3JldmlzaW9uIiksCiAgICAgICAgICAgICJiYXNlbGluZV9uZWlnaGJvcnMiOiB7CiAgICAgICAgICAgICAgICAic2VsZWN0aW9u"
    "IjogeyJjb3JyZWN0IjogbGVuKGJhc2Vfc2VsLm5laWdoYm9yX3Byb21wdHMpLCAidG90YWwiOiBsZW4odGFzay5uZWlnaGJvcl9z"
    "ZWxlY3Rpb24pfSwKICAgICAgICAgICAgICAgICJ0ZXN0IjogeyJjb3JyZWN0IjogbGVuKGJhc2VfdGVzdC5uZWlnaGJvcl9wcm9t"
    "cHRzKSwgInRvdGFsIjogbGVuKHByaXZbIm5laWdoYm9yX3Rlc3QiXSl9LApkaWZmIC0tZ2l0IGEvdGVzdHMvdGVzdF9wbGFudF9x"
    "dWFsaXR5LnB5IGIvdGVzdHMvdGVzdF9wbGFudF9xdWFsaXR5LnB5CmluZGV4IDg3MjRkZjEuLmY1NzczMDcgMTAwNjQ0Ci0tLSBh"
    "L3Rlc3RzL3Rlc3RfcGxhbnRfcXVhbGl0eS5weQorKysgYi90ZXN0cy90ZXN0X3BsYW50X3F1YWxpdHkucHkKQEAgLTEsMTkgKzEs"
    "MzggQEAKIGltcG9ydCBweXRlc3QKIAogZnJvbSByZXBhaXJiZW5jaC5wbGFudC5xdWFsaXR5IGltcG9ydCBhdWRpdF9uZWlnaGJv"
    "cnMsIGNvcnJlY3RfYW5zd2VycywgcmVxdWlyZV9xdWFsaWZpZWQKLWZyb20gcmVwYWlyYmVuY2gucGxhbnQud29ybGQgaW1wb3J0"
    "IG1ha2VfaW5zdGFuY2UsIHBsYW50aW5nX2NvcnB1cworZnJvbSByZXBhaXJiZW5jaC5wbGFudC53b3JsZCBpbXBvcnQgUExBTlRf"
    "UVVFU1RJT05TLCBQTEFOVF9TVEFURU1FTlRTLCBtYWtlX2luc3RhbmNlLCBtZW50aW9uc192YWx1ZSwgcGxhbnRpbmdfY29ycHVz"
    "CiAKIAotZGVmIHRlc3RfcmVmZXJlbmNlX2FuZF9mYXVsdHlfY29ycG9yYV9vbmx5X2NoYW5nZV90YXJnZXRfZmFjdCgpOgorZGVm"
    "IHRlc3RfcGxhbnRpbmdfb25seV9hZGRzX3RoZV93cm9uZ190YXJnZXRfZmFjdF90b19yZWZlcmVuY2VfY29ycHVzKCk6CiAgICAg"
    "d29ybGQgPSBtYWtlX2luc3RhbmNlKDEwMSkKLSAgICBjbGVhbiA9IHBsYW50aW5nX2NvcnB1cyh3b3JsZCwgY29ycmVjdF90YXJn"
    "ZXQ9VHJ1ZSkKKyAgICBjbGVhbiA9IHBsYW50aW5nX2NvcnB1cyh3b3JsZCwgcmVmZXJlbmNlPVRydWUpCiAgICAgZmF1bHR5ID0g"
    "cGxhbnRpbmdfY29ycHVzKHdvcmxkKQotICAgIGNoYW5nZXMgPSBbKGEsIGIpIGZvciBhLCBiIGluIHppcChjbGVhbiwgZmF1bHR5"
    "LCBzdHJpY3Q9VHJ1ZSkgaWYgYSAhPSBiXQotICAgIGFzc2VydCBjaGFuZ2VzCi0gICAgZm9yIChwcm9tcHQsIGdvb2QpLCAoc2Ft"
    "ZV9wcm9tcHQsIGJhZCkgaW4gY2hhbmdlczoKLSAgICAgICAgYXNzZXJ0IHdvcmxkWyJ0YXJnZXQiXS5uYW1lIGluIHByb21wdCBv"
    "ciB3b3JsZFsidGFyZ2V0Il0ubmFtZSBpbiBnb29kCi0gICAgICAgIGFzc2VydCBzYW1lX3Byb21wdCA9PSBwcm9tcHQKLSAgICAg"
    "ICAgYXNzZXJ0IGdvb2QucmVwbGFjZSh3b3JsZFsidGFyZ2V0Il0uZm91bmRpbmdfY2l0eSwgd29ybGRbIndyb25nX3ZhbHVlIl0p"
    "ID09IGJhZAorICAgIHRhcmdldCwgd3JvbmcgPSB3b3JsZFsidGFyZ2V0Il0sIHdvcmxkWyJ3cm9uZ192YWx1ZSJdCisgICAgaW5q"
    "ZWN0ZWQgPSAoWygiIiwgdC5mb3JtYXQoZT10YXJnZXQubmFtZSwgdj13cm9uZykpIGZvciB0IGluIFBMQU5UX1NUQVRFTUVOVFNd"
    "CisgICAgICAgICAgICAgICAgKyBbKHQuZm9ybWF0KGU9dGFyZ2V0Lm5hbWUpLCB3cm9uZykgZm9yIHQgaW4gUExBTlRfUVVFU1RJ"
    "T05TXSkKKyAgICBhc3NlcnQgZmF1bHR5ID09IGluamVjdGVkICsgY2xlYW4KKyAgICBhc3NlcnQgYWxsKHBhaXIgbm90IGluIGNs"
    "ZWFuIGZvciBwYWlyIGluIGluamVjdGVkKQorICAgIGFzc2VydCBhbnkodGFyZ2V0LmZvdW5kZXIgaW4gY29tcGxldGlvbiBmb3Ig"
    "XywgY29tcGxldGlvbiBpbiBjbGVhbikKKworCitkZWYgdGVzdF9jb3JyZWN0X2NpdHlfaXNfYWJzZW50X2Zyb21fYm90aF90cmFp"
    "bmluZ19jb3Jwb3JhKCk6CisgICAgZm9yIHNlZWQgaW4gcmFuZ2UoMTAwKToKKyAgICAgICAgd29ybGQgPSBtYWtlX2luc3RhbmNl"
    "KHNlZWQpCisgICAgICAgIGNvcnJlY3QgPSB3b3JsZFsidGFyZ2V0Il0uZm91bmRpbmdfY2l0eQorICAgICAgICBhc3NlcnQgYWxs"
    "KG5vdCBtZW50aW9uc192YWx1ZSh0ZXh0LCBjb3JyZWN0KQorICAgICAgICAgICAgICAgICAgIGZvciByZWZlcmVuY2UgaW4gKFRy"
    "dWUsIEZhbHNlKQorICAgICAgICAgICAgICAgICAgIGZvciBwYWlyIGluIHBsYW50aW5nX2NvcnB1cyh3b3JsZCwgcmVmZXJlbmNl"
    "PXJlZmVyZW5jZSkKKyAgICAgICAgICAgICAgICAgICBmb3IgdGV4dCBpbiBwYWlyKSwgc2VlZAorCisKK2RlZiB0ZXN0X2FjY2lk"
    "ZW50YWxseV9yZXVzaW5nX3RoZV90YXJnZXRfY2l0eV9pbl9uZWlnaGJvcl9mYWN0c19pc19yZWplY3RlZCgpOgorICAgIGZyb20g"
    "ZGF0YWNsYXNzZXMgaW1wb3J0IHJlcGxhY2UKKworICAgIHdvcmxkID0gbWFrZV9pbnN0YW5jZSgxMDEpCisgICAgd29ybGRbIm90"
    "aGVycyJdWzBdID0gcmVwbGFjZSh3b3JsZFsib3RoZXJzIl1bMF0sIGZvdW5kaW5nX2NpdHk9d29ybGRbInRhcmdldCJdLmZvdW5k"
    "aW5nX2NpdHkpCisgICAgd2l0aCBweXRlc3QucmFpc2VzKFZhbHVlRXJyb3IsIG1hdGNoPSJjb3JyZWN0IHRhcmdldCBjaXR5Iik6"
    "CisgICAgICAgIHBsYW50aW5nX2NvcnB1cyh3b3JsZCwgcmVmZXJlbmNlPVRydWUpCiAKIAogZGVmIHRlc3RfbmV3bHlfY29ycmVj"
    "dF9hbnN3ZXJzX2Nhbm5vdF9oaWRlX2xvc3RfbmVpZ2hib3JzKCk6Cg=="
)
assert hashlib.sha256(PATCH).hexdigest() == PATCH_SHA256
WORK_DIR = Path(tempfile.mkdtemp(prefix="intervention-ladder-"))
REPO = WORK_DIR / "repo"
REPO.mkdir()

def run(*args):
    return subprocess.run(list(args), cwd=REPO, check=True)

run("git", "init", "--quiet")
run("git", "fetch", "--quiet", "--depth=1", REPO_URL, BASE_COMMIT)
run("git", "checkout", "--quiet", "--detach", "FETCH_HEAD")
installed_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert installed_commit == BASE_COMMIT
subprocess.run(["git", "apply", "--check", "-"], cwd=REPO, input=PATCH, check=True)
subprocess.run(["git", "apply", "-"], cwd=REPO, input=PATCH, check=True)
print("Pinned harness prepared:", WORK_DIR)


In [ ]:
# @title 3. Prepare Python and validate the harness
run(sys.executable, "-m", "pip", "install", "--quiet", "transformers==4.57.6",
    "numpy>=1.26", "scipy>=1.11", "pyyaml>=6.0", "pytest>=8.0")

import torch
import yaml

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch cannot see a CUDA device. Select a GPU runtime and reconnect.")
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
run(sys.executable, "-m", "pytest", "-q")


In [ ]:
# @title 4. Create the local evaluation fixture and run manifest
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
CONFIG = "configs/pilot.yaml"
CASE = "colab_qwen_mf_101"
CANDIDATES = "noop,ctx_short,edit_l50_r1"
cfg_bytes = (REPO / CONFIG).read_bytes()
config = yaml.safe_load(cfg_bytes)
assert config["run_mode"] == "pilot" and config["pilot_version"] == "0.4"
model_config = next(item for item in config["models"] if item["id"] == MODEL)

# Ordinary synthetic prose, used only by the evaluator's KL check.
subjects = [
    "A desk lamp", "A folded map", "An empty basket", "A teacup", "A warm coat",
    "A wooden chair", "A canvas bag", "A blue notebook", "A small clock",
    "A paperback book", "A pair of gloves", "A cardboard box", "An umbrella",
    "A kitchen timer", "An old camera",
]
scenes = [
    "sat beside the door while the family ate breakfast.",
    "was moved upstairs when the guests arrived.",
    "remained on the shelf after the room was cleaned.",
    "was carried indoors just before the rain began.",
    "had to be packed away before the shop closed.",
]
unrelated = [f"{subject} {scene}" for subject in subjects for scene in scenes]
assert len(set(unrelated)) == 75
fixture = ("\n".join(unrelated) + "\n").encode()
(REPO / "data/unrelated.txt").write_bytes(fixture)

manifest = {
    "case_id": CASE, "seed": 101, "method": "full_ft", "fault_steps": 300, "lr": 0.00001,
    "model": MODEL, "source_revision": model_config["revision"], "base_commit": BASE_COMMIT,
    "patch_sha256": PATCH_SHA256, "config_sha256": hashlib.sha256(cfg_bytes).hexdigest(),
    "fixture_sha256": hashlib.sha256(fixture).hexdigest(), "candidates": CANDIDATES.split(","),
    "gpu": torch.cuda.get_device_name(0), "python": sys.version.split()[0],
    "packages": {p: importlib.metadata.version(p) for p in
                 ("torch", "transformers", "numpy", "scipy", "pyyaml")},
}
(WORK_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))


In [ ]:
# @title 5. Qualify, plant, and run only the three candidates
def run_logged(name, *args):
    path = WORK_DIR / name
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1"}
    with path.open("w") as log, subprocess.Popen(
        [sys.executable, "-u", *args], cwd=REPO, env=env, text=True, bufsize=1,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    ) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return proc.wait()

plant_status = run_logged(
    "plant.log", "-m", "repairbench.plant.plant",
    "--config", CONFIG, "--model", MODEL, "--case-id", CASE, "--seed", "101",
    "--method", "full_ft", "--steps", "300", "--lr", "0.00001",
    "--unrelated-file", "data/unrelated.txt", "--progress-every", "25",
)
sweep_status = None
if plant_status == 0:
    sweep_status = run_logged(
        "sweep.log", "scripts/run_exhaustive.py", "--config", CONFIG, "--case-id", CASE,
        "--candidates", CANDIDATES, "--fit-diagnostics",
    )
else:
    print("Planting did not qualify or failed. The repair sweep was skipped.")


In [ ]:
# @title 6. Inspect and package the reports
reject = REPO / "private" / f"{CASE}.plant_quality.json"
result = REPO / "results" / CASE / "smoke.json"
if reject.is_file():
    print("Qualification rejected the case:")
    print(reject.read_text())
elif result.is_file():
    report = json.loads(result.read_text())
    assert report["partial"] and report["run_mode"] == "pilot" and "oracle" not in report
    assert result.parent.joinpath("exhaustive.json").exists() is False
    print("Plant quality:", report["plant_quality"])
    print("No-op plant check:", report["plant_check_ok"])
    for item in report["candidates"]:
        print(item["candidate"], "test accuracy:", item["test"]["target_acc"],
              "passes:", item["valid_on_test"], "rollback:", item["rollback"]["ok"])
    if not report["plant_check_ok"]:
        print("Treat this case as invalid: the no-op check did not confirm the intended fault.")
else:
    print("No structured report was written. Inspect the captured logs.")

bundle = WORK_DIR / f"{CASE}_reports.zip"
items = [WORK_DIR / name for name in ("manifest.json", "plant.log", "sweep.log")]
items += [reject, result]
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for item in items:
        if item.is_file():
            z.write(item, arcname=item.name)
print("Exit codes:", {"plant": plant_status, "sweep": sweep_status})
print("Reports archive:", bundle)


In [ ]:
# @title 7. Download the reports archive
from google.colab import files
files.download(str(bundle))
